In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("datasets/maternal_care_datasets_augmented.csv")  # change to your dataset path

# =========================
# 2. DEFINE FEATURES & TARGET
# =========================
target = "RiskLevel"  # change if needed

X = df.drop(columns=[target])
y = df[target]

# Save feature names
feature_names = list(X.columns)

# =========================
# 3. ENCODE TARGET
# =========================
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Save label mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

# =========================
# 4. PREPROCESSING
# =========================
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ]
)

# =========================
# 5. MODEL (RandomForest)
# =========================
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)

# Full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# =========================
# 6. TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# =========================
# 7. TRAIN MODEL
# =========================
pipeline.fit(X_train, y_train)

# =========================
# 8. EVALUATE
# =========================
y_pred = pipeline.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# =========================
# 9. EXTRACT COMPONENTS
# =========================
trained_model = pipeline.named_steps['model']
trained_preprocessor = pipeline.named_steps['preprocessor']

# =========================
# 10. SAVE EVERYTHING
# =========================
joblib.dump(trained_model, "model.pkl")
joblib.dump(trained_preprocessor, "preprocessor.pkl")

with open("feature_names.json", "w") as f:
    json.dump(feature_names, f)

with open("label_mapping.json", "w") as f:
    json.dump(label_mapping, f)

print("\n✅ Model, preprocessor, and metadata saved successfully!")